# 02 · Chunking and retrieval — finding the right course material

**Step 2 of the Study Assistant project.** Given a student's question, find the pieces of course material that answer it. We compare four retrieval methods:

| Method | Idea | From class |
|---|---|---|
| **TF-IDF** | keyword overlap, rare words weigh more | `RAG_primer_fast.ipynb` |
| **BM25** | the standard search-engine keyword ranking (better length handling than TF-IDF) | — |
| **Dense** | sentence embeddings (`all-MiniLM-L6-v2`), compare meaning not words | `campus_help_desk_rag.ipynb`, `research_ingest.py` |
| **Hybrid** | BM25 + dense merged with Reciprocal Rank Fusion | — |

Code: `src/study_assistant/chunking.py`, `retrieval.py`, `retrieval_eval.py`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)

from study_assistant.ingest import load_documents
from study_assistant.chunking import chunk_documents, count_tokens, MAX_TOKENS
from study_assistant.retrieval import build_retriever, Embedder
from study_assistant.retrieval_eval import load_queries, evaluate, summarize

docs = load_documents()
chunks = chunk_documents(docs)
print(f"{len(docs)} documents -> {len(chunks)} chunks")

507 documents -> 1202 chunks


## 1. Chunking

The embedding model reads at most **256 tokens** (254 + 2 special tokens). Anything longer is silently cut off, so the end of a long notebook section would be invisible to search. Slides fit, notebook sections mostly don't:

In [2]:
tok = pd.DataFrame({"type": d.source_type, "tokens": count_tokens(d.text)} for d in docs if not d.extra["low_content"])
tok.groupby("type").tokens.describe(percentiles=[.5, .9])[["count", "50%", "90%", "max"]].astype(int).assign(
    over_limit=tok.groupby("type").tokens.apply(lambda s: int((s > MAX_TOKENS).sum())))

,count,50%,90%,max,over_limit
type,,,,,
activity,11,221,267,1204,3
guide,4,624,651,654,4
notebook,249,457,1382,4845,185
slides,200,105,178,312,1


So documents are split on paragraph / code-block boundaries (never mid-sentence or mid-function), with each chunk sized by the model's own tokenizer. Every chunk also gets a **header** `title › section` in the text that is embedded, so a piece of code from the middle of a notebook still says what it is about.

Checks: no chunk exceeds the limit, and no text is lost (also in `tests/test_chunking.py`).

In [3]:
lens = np.array([count_tokens(c.embed_text) for c in chunks])
print(f"max tokens per chunk: {lens.max()} (limit {MAX_TOKENS}), median: {int(np.median(lens))}")
pd.Series([c.source_type for c in chunks]).value_counts().rename("chunks")

max tokens per chunk: 254 (limit 254), median: 203


notebook    971
slides      201
activity     18
guide        12
Name: chunks, dtype: int64

In [4]:
c = next(c for c in chunks if c.doc_id == "Module_4_Practical_VAE.ipynb::section-15")
print(f"{c.chunk_id}  (part {c.part} of {c.n_parts})\n")
print(c.embed_text[:700])

Module_4_Practical_VAE.ipynb::section-15#1  (part 1 of 4)

Module 4: Practical - Variational Autoencoders › Variational Autoencoder (VAE) Architecture

## Variational Autoencoder (VAE) Architecture

The idea behind VAE is for the encoder to map to a distribution, rather than an exact latent variable. So the encoder network now outputs two vectors, the mean and the log variance of a normal distribution.
In turn, the decoder, first samples from this distribution and then uses the same decoder as before to map the sampled latent variable back to the original image space.


## 2. Test questions

`eval/retrieval_queries.jsonl` has 35 questions. For each one we first **grepped the knowledge base** to confirm which lecture (or lab notebook) actually covers it, so the answer key is verified, not guessed.

- **20 keyword questions** use the topic's own term ("Bellman equation", "BLEU", "LoRA").
- **10 paraphrased questions** describe the idea *without* the term, like a student who doesn't know the name yet ("Why can't we backpropagate through random sampling?" → reparameterization trick, Lecture 5).
- **5 out-of-scope questions**, including a near-miss: *mode collapse* sounds like GAN material, but the course never mentions it.

A retrieved chunk is correct if it comes from the gold lecture/file. **hit@k** = share of questions with a correct chunk in the top k; **MRR** rewards ranking it first.

In [5]:
queries = load_queries()
emb = Embedder()  # embeddings are computed once (~10 s) and cached in data/cache/
results = pd.concat([evaluate(build_retriever(m, chunks, emb), queries) for m in ["tfidf", "bm25", "dense", "hybrid"]])
results["set"] = results.id.str[0].map({"r": "keyword", "p": "paraphrase", "o": "out of scope"})
pd.DataFrame(queries)[["id", "query"]].set_index("id").T

W1124 15:44:08.983000 72208 /Users/apple/Desktop/ /Columbia/Sem 3/Assignments/Applied Gen AI/Final Project/.venv.nosync/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20034.01it/s]

id,r01,r02,r03,r04,r05,r06,r07,r08,r09,r10,...,p01,p02,p03,p04,p05,p06,p07,p08,p09,p10
query,Why do we need the reparameterization trick when training a VAE?,What does the KL divergence term in the VAE loss do?,How is the GAN discriminator trained and what loss does it use?,How do LSTMs differ from simple RNNs?,How does a normalizing flow compute the probability of a data point?,What is Real NVP?,How does a diffusion model add noise to an image and then learn to remove it?,How do energy-based models generate new samples?,Why does a transformer need positional encodings?,What is the Bellman equation for the value function?,...,"Why can't we backpropagate through random sampling in an autoencoder, and how do we ge...","If attention looks at all the words at once, how does the model know their order?",How do you teach a chatbot to prefer the answers people like?,How can I adapt a big language model cheaply by only training a few small extra matrices?,How does an agent figure out which actions pay off in the long run rather than right a...,How do two networks compete with each other to produce realistic fake images?,How do we turn words into vectors so that similar words end up close together?,How can we generate images by starting from pure static and cleaning it up step by step?,How do you get an LLM to grade other models' answers automatically?,How does a model shrink an image's resolution while keeping the strongest features?


## 3. Results

In [6]:
print("Keyword questions (20)"); display(summarize(results[results.set == "keyword"]))
print("Paraphrased questions (10)"); display(summarize(results[results.set == "paraphrase"]))
print("All in-scope questions (30)"); display(summarize(results))

Keyword questions (20)


,hit@1,hit@3,hit@5,MRR@10
tfidf,1.00,1.00,1.00,1.000
bm25,1.00,1.00,1.00,1.000
dense,0.90,0.95,0.95,0.933
hybrid,0.95,1.00,1.00,0.975


Paraphrased questions (10)


,hit@1,hit@3,hit@5,MRR@10
tfidf,0.3,0.5,0.5,0.424
bm25,0.5,0.7,0.8,0.592
dense,0.7,0.9,0.9,0.800
hybrid,0.7,0.7,0.7,0.712


All in-scope questions (30)


,hit@1,hit@3,hit@5,MRR@10
tfidf,0.767,0.833,0.833,0.808
bm25,0.833,0.900,0.933,0.864
dense,0.833,0.933,0.933,0.889
hybrid,0.867,0.900,0.900,0.888


Rank of the first correct chunk for every paraphrased question (`NaN` = not in the top 10):

In [7]:
results[results.set == "paraphrase"].pivot(index="id", columns="method", values="first_relevant_rank")[["tfidf", "bm25", "dense", "hybrid"]]

method,tfidf,bm25,dense,hybrid
id,,,,
p01,1.0,1.0,1.0,1.0
p02,9.0,1.0,1.0,1.0
p03,NaN,NaN,6.0,NaN
p04,8.0,3.0,2.0,NaN
p05,1.0,1.0,1.0,1.0
p06,2.0,3.0,1.0,1.0
p07,1.0,1.0,1.0,1.0
p08,2.0,1.0,1.0,1.0
p09,NaN,4.0,1.0,1.0


**Reading the results**

- With the key term in the question, **keyword search is perfect**: the term only appears in the right lecture.
- Without it, **keyword methods fall apart** (TF-IDF finds 5 of 10) while **dense retrieval finds 9 of 10**: it matches meaning.
- **Hybrid is not automatically best.** Rank fusion rewards chunks that *both* lists rank fairly high. In p04 (LoRA) BM25 and dense each found a *different part* of the same notebook section (part 3 vs part 1 of the introduction), so each part got only one vote, and chunks that both lists half-liked won, pushing the LoRA notebook out of the top 10:

In [8]:
q = "How can I adapt a big language model cheaply by only training a few small extra matrices?"
for m in ["bm25", "dense", "hybrid"]:
    print(f"--- {m}")
    for h in build_retriever(m, chunks, emb).search(q, k=4):
        print(f"  {h.rank}. {h.chunk.citation[:80]}  [part {h.chunk.part}/{h.chunk.n_parts}]")

--- bm25
  1. Module 3 – Applications of Generative AI, slide 14  [part 1/1]
  2. Module 11 – Application of Reinforcement Learning to Generative AI, slide 3  [part 1/1]
  3. Parameter-Efficient Fine-Tuning with LoRA, cell 1 (Introduction)  [part 3/5]
  4. Module 10 — Applying Reinforcement Learning to Language, cells 6-7 (Visualizing   [part 1/6]
--- dense
  1. Module 3 – Applications of Generative AI, slide 14  [part 1/1]
  2. Parameter-Efficient Fine-Tuning with LoRA, cell 29 (8. Summary and exercises)  [part 1/4]
  3. Parameter-Efficient Fine-Tuning with LoRA, cell 1 (Introduction)  [part 1/5]
  4. Module 2: Practice 2 - Word Sampling, cells 10-11 (Bigram Probabilities › While   [part 1/3]
--- hybrid
  1. Module 3 – Applications of Generative AI, slide 14  [part 1/1]
  2. Module 11 – Application of Reinforcement Learning to Generative AI, slide 3  [part 1/1]
  3. Module 4 – Modern Deep Learning Architectures Part 1, slide 5  [part 1/1]
  4. Fine-tuning T5 for Translation — and Meas

A possible fix, to test in Step 6: merge the votes of chunks from the same section before fusing.

We are **not** tuning the methods on these 30 questions: we wrote them ourselves while building the system, so tuning would just fit our own test. The final choice is made on the separate 50-question evaluation in Step 5.

## 4. Can a similarity score tell "covered" from "not covered"?

The assistant must say *"this isn't covered in the course"* instead of guessing. The simplest idea is a threshold on the best cosine similarity (like `DISTANCE_THRESHOLD` in `campus_help_desk_rag.ipynb`):

In [9]:
d = results[results.method == "dense"]
print("best similarity for out-of-scope questions:")
display(d[d.set == "out of scope"][["query", "max_dense_score", "top_hit"]].round(2))
inscope = d[d.set != "out of scope"].max_dense_score
print(f"in-scope questions: lowest best-similarity = {inscope.min():.2f}; "
      f"{(inscope < d[d.set == 'out of scope'].max_dense_score.max()).sum()} of {len(inscope)} score below the mode-collapse near-miss")

best similarity for out-of-scope questions:


,query,max_dense_score,top_hit
20,What is the capital of France?,0.36,"LLM / Agent Evaluations, cells 9-10 (Tier 3 — Semantic similarity (sentence embeddings))"
21,How does a Kalman filter work?,0.29,"Module 10 – Basics of Reinforcement Learning, slide 13"
22,What is mode collapse in GANs and how do I fix it?,0.48,"Module 9 – Transformer Architecture and Advanced GAN, slide 14"
23,How do support vector machines find the maximum margin?,0.31,"Module 3: Practical 1 - Fully Connected Neural Networks, cells 6-8 (Loss Functions › W..."
24,What is the time complexity of quicksort?,0.29,"Parameter-Efficient Fine-Tuning with LoRA, cells 14-16 (4. Applying LoRA)"


in-scope questions: lowest best-similarity = 0.37; 7 of 30 score below the mode-collapse near-miss


**No threshold works.** "Capital of France" (0.36) scores almost as high as the weakest real question (0.37), and the near-miss *mode collapse* (0.48) beats seven real questions, because it shares vocabulary with the GAN lecture.

So the similarity score can only filter out obvious off-topic questions; deciding whether the retrieved material *actually answers* the question needs the LLM. That is why the assistant in Step 4 gets a **relevance-check step** before answering.

## 5. Summary

| Finding | Consequence |
|---|---|
| 171 documents exceed the 256-token model limit | Token-aware chunking: 1,202 chunks, none over the limit, nothing lost |
| Keyword search is perfect when the question uses the course's term, weak otherwise | Students often don't know the term; meaning-based retrieval matters |
| Dense: best on paraphrases (90% hit@5); hybrid did not beat it here | All four methods kept; the default is chosen on the Step 5 evaluation |
| Similarity scores overlap for in-scope and out-of-scope questions | Step 4 adds an LLM relevance check instead of a fixed threshold |

**Limitations:** 30 in-scope questions is small (one question = 3–10 percentage points), and relevance is judged at the lecture level, not the exact slide.